# EcoVision - Clasificacion Automatica de Residuos Reciclables

**Entrega 3 - Vision Artificial**
Docente: Cristian Mateo Hernandez Gonzalez

**Start-Up:** EcoVision
**Integrantes:** Dorian Argel, Simón García

---

## El problema

La mala separacion de residuos en el punto donde se generan (oficinas, campus, centros comerciales, pequenos centros de acopio) sigue siendo mayormente manual y propensa a error humano.


## 0. Configuracion e importaciones

Antes de correr el notebook: descarga el dataset de Kaggle, descomprimelo, y ajusta `RUTA_DATASET` en la siguiente celda a la carpeta donde quedo. No importa cuantos niveles de carpetas anidadas traiga el `.zip` -- el notebook busca cada carpeta de clase recursivamente.


In [ ]:
import os
import time

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops

# ============================================================
# CONFIGURACION -- ajustar antes de correr
# ============================================================

# Carpeta raiz del dataset "Garbage Classification" descargado de Kaggle
RUTA_DATASET = "C:/Users/Usuario/Downloads/garbage-classification"

# Carpeta donde se guardaran los .npz con las caracteristicas extraidas
OUTPUT_DIR = "C:/Users/Usuario/Downloads/features_reciclaje"

CLASES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

TAMANO_IMAGEN = (128, 128)
EXTENSIONES = ('.jpg', '.jpeg', '.png', '.bmp')

# Limite de imagenes por clase (GLCM es el descriptor mas costoso de calcular)
MAX_IMAGENES_POR_CLASE = 300

# Dos clases usadas para la visualizacion detallada de descriptores
CLASE_EJEMPLO_A = 'cardboard'
CLASE_EJEMPLO_B = 'plastic'

PROMEDIO_METRICAS = 'weighted'  # 'weighted' (recomendado para clases desbalanceadas) o 'macro'


## 1. Extraccion de caracteristicas

### 1.1 Carga y balanceo del dataset

Se busca la carpeta de cada clase dentro de `RUTA_DATASET`, se cuenta cuantas imagenes hay disponibles en cada una, y se balancea el dataset usando el minimo comun (para que ninguna clase domine el entrenamiento).


In [ ]:
def encontrar_carpeta_clase(raiz, nombre_clase):
    """Busca recursivamente la carpeta cuyo nombre coincide con 'nombre_clase'."""
    for carpeta_actual, subcarpetas, _ in os.walk(raiz):
        for sub in subcarpetas:
            if sub.lower() == nombre_clase.lower():
                return os.path.join(carpeta_actual, sub)
    return None


def cargar_imagenes_validas(path, n):
    archivos = sorted(f for f in os.listdir(path) if f.lower().endswith(EXTENSIONES))
    imgs = []
    for f in archivos:
        img = cv2.imread(os.path.join(path, f))
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            imgs.append(img)
        if len(imgs) == n:
            break
    return imgs


def normalizar_imagenes(imgs, size=TAMANO_IMAGEN):
    imgs_norm = []
    for img in imgs:
        if img is not None:
            img_resized = cv2.resize(img, size)
            img_norm = img_resized.astype("float32") / 255.0
            imgs_norm.append(img_norm)
    return np.array(imgs_norm)


rutas_clase, conteos = {}, {}
for clase in CLASES:
    ruta = encontrar_carpeta_clase(RUTA_DATASET, clase)
    if ruta is None:
        raise FileNotFoundError(f"No se encontro la carpeta de la clase '{clase}' en '{RUTA_DATASET}'.")
    rutas_clase[clase] = ruta
    conteos[clase] = len([f for f in os.listdir(ruta) if f.lower().endswith(EXTENSIONES)])

print("Imagenes disponibles por clase:")
for clase in CLASES:
    print(f"  {clase:10s}: {conteos[clase]}")

n_por_clase = min(min(conteos.values()), MAX_IMAGENES_POR_CLASE)
print(f"\nUsando {n_por_clase} imagenes por clase (dataset balanceado).")


In [ ]:
imagenes_por_clase, etiquetas = [], []
for idx_clase, clase in enumerate(CLASES):
    imgs_clase = cargar_imagenes_validas(rutas_clase[clase], n_por_clase)
    imgs_norm = normalizar_imagenes(imgs_clase)
    imagenes_por_clase.append(imgs_norm)
    etiquetas.extend([idx_clase] * len(imgs_norm))

X = np.concatenate(imagenes_por_clase, axis=0)
y = np.array(etiquetas)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Dataset listo -> Train: {X_train.shape}, Validacion: {X_val.shape}")


### 1.2 Descriptores: HOG, LBP y GLCM

- **HOG** (Histogram of Oriented Gradients): captura la forma/silueta del objeto a partir de la   orientacion de los bordes.
- **LBP** (Local Binary Patterns): captura micro-textura comparando cada pixel con sus vecinos.
- **GLCM** (Gray-Level Co-occurrence Matrix): descriptor estadistico de textura (contraste,   homogeneidad, energia, correlacion, etc.).


In [ ]:
def extraer_hog(imagenes):
    hog_features = []
    for img in imagenes:
        img_gray = cv2.cvtColor((img * 255).astype("uint8"), cv2.COLOR_RGB2GRAY)
        features = hog(img_gray, orientations=9, pixels_per_cell=(8, 8),
                        cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)
        hog_features.append(features)
    return np.array(hog_features)


def extraer_lbp(imagenes, P=8, R=1):
    lbp_features = []
    for img in imagenes:
        img_gray = cv2.cvtColor((img * 255).astype("uint8"), cv2.COLOR_RGB2GRAY)
        lbp = local_binary_pattern(img_gray, P=P, R=R, method="uniform")
        (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, P + 3), range=(0, P + 2))
        hist = hist.astype("float32")
        hist /= (hist.sum() + 1e-7)
        lbp_features.append(hist)
    return np.array(lbp_features)


def extraer_glcm(imagenes, distances=[1, 3], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4]):
    glcm_features = []
    propiedades = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
    for img in imagenes:
        img_gray = cv2.cvtColor((img * 255).astype("uint8"), cv2.COLOR_RGB2GRAY)
        img_gray_q = (img_gray // 4).astype(np.uint8)
        glcm = graycomatrix(img_gray_q, distances=distances, angles=angles, levels=64,
                             symmetric=True, normed=True)
        vector_img = []
        for prop in propiedades:
            vector_img.extend(graycoprops(glcm, prop).ravel())
        glcm_features.append(vector_img)
    return np.array(glcm_features)


print("Extrayendo HOG...")
X_train_hog, X_val_hog = extraer_hog(X_train), extraer_hog(X_val)

print("Extrayendo LBP...")
X_train_lbp, X_val_lbp = extraer_lbp(X_train), extraer_lbp(X_val)

print("Extrayendo GLCM...")
X_train_glcm, X_val_glcm = extraer_glcm(X_train), extraer_glcm(X_val)

print("\nDimensiones resultantes:")
print(f"HOG  -> Train: {X_train_hog.shape}")
print(f"LBP  -> Train: {X_train_lbp.shape}")
print(f"GLCM -> Train: {X_train_glcm.shape}")


### 1.3 Visualizacion de los descriptores

In [ ]:
fig0, axes0 = plt.subplots(1, len(CLASES), figsize=(3 * len(CLASES), 3))
for i, clase in enumerate(CLASES):
    idx_ejemplo = np.where(y_val == i)[0]
    if len(idx_ejemplo) > 0:
        axes0[i].imshow(X_val[idx_ejemplo[0]])
    axes0[i].set_title(clase)
    axes0[i].axis('off')
plt.suptitle("Una muestra de cada clase del dataset de reciclaje")
plt.tight_layout()
plt.show()


In [ ]:
idx_a = np.where(y_val == CLASES.index(CLASE_EJEMPLO_A))[0][0]
idx_b = np.where(y_val == CLASES.index(CLASE_EJEMPLO_B))[0][0]

img_a_gray = cv2.cvtColor((X_val[idx_a] * 255).astype("uint8"), cv2.COLOR_RGB2GRAY)
img_b_gray = cv2.cvtColor((X_val[idx_b] * 255).astype("uint8"), cv2.COLOR_RGB2GRAY)

_, hog_a_vis = hog(img_a_gray, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)
_, hog_b_vis = hog(img_b_gray, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)

lbp_a_vis = local_binary_pattern(img_a_gray, P=8, R=1, method="uniform")
lbp_b_vis = local_binary_pattern(img_b_gray, P=8, R=1, method="uniform")

a_q = (img_a_gray // 4).astype(np.uint8)
b_q = (img_b_gray // 4).astype(np.uint8)
glcm_a_vis = graycomatrix(a_q, distances=[1], angles=[0], levels=64, symmetric=True, normed=True)[:, :, 0, 0]
glcm_b_vis = graycomatrix(b_q, distances=[1], angles=[0], levels=64, symmetric=True, normed=True)[:, :, 0, 0]

fig1, axes1 = plt.subplots(4, 2, figsize=(11, 13))

axes1[0, 0].imshow(X_val[idx_a]); axes1[0, 0].set_title(f"{CLASE_EJEMPLO_A} (Original)"); axes1[0, 0].axis('off')
axes1[0, 1].imshow(X_val[idx_b]); axes1[0, 1].set_title(f"{CLASE_EJEMPLO_B} (Original)"); axes1[0, 1].axis('off')

axes1[1, 0].imshow(hog_a_vis, cmap='gray'); axes1[1, 0].set_title(f"HOG ({CLASE_EJEMPLO_A})"); axes1[1, 0].axis('off')
axes1[1, 1].imshow(hog_b_vis, cmap='gray'); axes1[1, 1].set_title(f"HOG ({CLASE_EJEMPLO_B})"); axes1[1, 1].axis('off')

axes1[2, 0].imshow(lbp_a_vis, cmap='gray'); axes1[2, 0].set_title(f"LBP ({CLASE_EJEMPLO_A})"); axes1[2, 0].axis('off')
axes1[2, 1].imshow(lbp_b_vis, cmap='gray'); axes1[2, 1].set_title(f"LBP ({CLASE_EJEMPLO_B})"); axes1[2, 1].axis('off')

axes1[3, 0].imshow(glcm_a_vis, cmap='viridis'); axes1[3, 0].set_title(f"GLCM ({CLASE_EJEMPLO_A})")
axes1[3, 1].imshow(glcm_b_vis, cmap='viridis'); axes1[3, 1].set_title(f"GLCM ({CLASE_EJEMPLO_B})")

plt.tight_layout()
plt.show()


## 2. Reduccion / Normalizacion (Fusion + Z-score)

Se concatenan los tres descriptores en un solo vector por imagen, y se normaliza con `StandardScaler` (Z-score): se resta la media y se divide por la desviacion estandar de cada caracteristica, calculadas **solo** con el conjunto de entrenamiento (el `scaler` se ajusta con `fit_transform` en train y solo se aplica con `transform` en validacion, para evitar fuga de informacion del conjunto de prueba).


In [ ]:
X_train_fusion = np.hstack((X_train_hog, X_train_lbp, X_train_glcm))
X_val_fusion = np.hstack((X_val_hog, X_val_lbp, X_val_glcm))

scaler = StandardScaler()
X_train_fusion_scaled = scaler.fit_transform(X_train_fusion)
X_val_fusion_scaled = scaler.transform(X_val_fusion)

print(f"Vector Fusionado y Escalado -> Train: {X_train_fusion_scaled.shape}, Val: {X_val_fusion_scaled.shape}")


In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
clases_arr = np.array(CLASES)

def escalar_y_guardar(nombre, X_tr, X_va):
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_va_sc = sc.transform(X_va)
    np.savez_compressed(os.path.join(OUTPUT_DIR, f"features_{nombre}.npz"),
                         X_train=X_tr_sc, X_val=X_va_sc,
                         y_train=y_train, y_val=y_val, clases=clases_arr)

escalar_y_guardar("hog", X_train_hog, X_val_hog)
escalar_y_guardar("lbp", X_train_lbp, X_val_lbp)
escalar_y_guardar("glcm", X_train_glcm, X_val_glcm)

np.savez_compressed(os.path.join(OUTPUT_DIR, "features_fusion.npz"),
                     X_train=X_train_fusion_scaled, X_val=X_val_fusion_scaled,
                     y_train=y_train, y_val=y_val, clases=clases_arr)

print("Caracteristicas guardadas exitosamente en:", OUTPUT_DIR)


## 3. Clasificacion y Benchmarking

Se entrenan los tres clasificadores pedidos (k-NN, SVM, Random Forest) sobre cada uno de los cuatro conjuntos de caracteristicas (HOG, LBP, GLCM, Fusion) -- 12 combinaciones en total -- y se comparan con Accuracy, Precision, Recall y F1-Score. Como son 6 clases (no 2), las metricas usan `average='weighted'`, que pondera por cuantas imagenes tiene cada clase en validacion.


In [ ]:
FEATURE_FILES = {
    'HOG': 'features_hog.npz',
    'LBP': 'features_lbp.npz',
    'GLCM': 'features_glcm.npz',
    'FUSION (HOG+LBP+GLCM)': 'features_fusion.npz'
}

models = {
    'k-NN (k=5)': KNeighborsClassifier(n_neighbors=5, metric='euclidean'),
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

results = []
matrices_confusion = {}
clases = None

print("Iniciando Benchmark de Modelos y Descriptores...\n")

for feat_name, file_name in FEATURE_FILES.items():
    data = np.load(os.path.join(OUTPUT_DIR, file_name), allow_pickle=True)
    X_tr, X_va = data['X_train'], data['X_val']
    y_tr, y_va = data['y_train'], data['y_val']
    if clases is None and 'clases' in data:
        clases = list(data['clases'])

    for model_name, model in models.items():
        t0 = time.time()
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_va)
        t_exec = time.time() - t0

        results.append({
            'Descriptor': feat_name,
            'Modelo': model_name,
            'Dimensiones': X_tr.shape[1],
            'Accuracy': accuracy_score(y_va, y_pred),
            'Precision': precision_score(y_va, y_pred, average=PROMEDIO_METRICAS, zero_division=0),
            'Recall': recall_score(y_va, y_pred, average=PROMEDIO_METRICAS, zero_division=0),
            'F1-Score': f1_score(y_va, y_pred, average=PROMEDIO_METRICAS, zero_division=0),
            'Tiempo (s)': t_exec
        })
        matrices_confusion[(feat_name, model_name)] = confusion_matrix(y_va, y_pred)

df_results = pd.DataFrame(results)
print("TABLA CONSOLIDADA DE BENCHMARK (CLASIFICACION DE RESIDUOS RECICLABLES - 6 CLASES)")
df_results


In [ ]:
plt.figure(figsize=(12, 6))
df_pivot = df_results.pivot(index='Descriptor', columns='Modelo', values='F1-Score')
ax = df_pivot.plot(kind='bar', figsize=(11, 6), colormap='Set2', edgecolor='black')
plt.title("Comparativa de Desempeno (F1-Score Ponderado) segun Extractor y Algoritmo ML")
plt.ylabel("F1-Score"); plt.xlabel("Extractor de Caracteristicas")
plt.ylim(0, 1.05); plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title="Algoritmos ML", loc='lower right')
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 5), textcoords='offset points',
                     fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()


### Mejor combinacion y matriz de confusion

Con 6 clases el F1 global no dice *que* confunde el modelo -- la matriz de confusion es la que muestra, por ejemplo, si el sistema confunde vidrio con plastico, un dato clave para el guion del video-pitch.


In [ ]:
mejor_fila = df_results.loc[df_results['F1-Score'].idxmax()]
mejor_combo = (mejor_fila['Descriptor'], mejor_fila['Modelo'])
mejor_matriz = matrices_confusion[mejor_combo]

print(f"Mejor combinacion: {mejor_combo[0]} + {mejor_combo[1]} "
      f"(F1-Score ponderado = {mejor_fila['F1-Score']:.3f})")

plt.figure(figsize=(7, 6))
plt.imshow(mejor_matriz, cmap='Blues')
plt.title(f"Matriz de Confusion - {mejor_combo[0]} + {mejor_combo[1]}")
plt.colorbar(label="Numero de imagenes")
plt.xticks(range(len(clases)), clases, rotation=45, ha='right')
plt.yticks(range(len(clases)), clases)
plt.xlabel("Clase predicha"); plt.ylabel("Clase real")
for i in range(mejor_matriz.shape[0]):
    for j in range(mejor_matriz.shape[1]):
        color = "white" if mejor_matriz[i, j] > mejor_matriz.max() / 2 else "black"
        plt.text(j, i, str(mejor_matriz[i, j]), ha="center", va="center", color=color)
plt.tight_layout()
plt.show()


## Conclusiones

- La combinacion con mejor desempeno fue la **fusion de HOG + LBP + GLCM con SVM (RBF)**, con un F1-Score ponderado de **0.507** en el conjunto de validacion: cada descriptor aporta informacion distinta (forma, micro-textura, estadistica de textura) y combinados generalizan mejor que cualquiera por separado.
- **k-NN** quedo consistentemente ultimo en las cuatro combinaciones de descriptores: con vectores de alta dimension (HOG y Fusion superan las 8000 caracteristicas) la distancia euclidiana deja de ser tan informativa (maldicion de la dimensionalidad).
- Un F1 de 0.507 es bastante superior al azar para 6 clases (~0.167), lo que confirma que el sistema aprendio un patron real, aunque todavia hay margen de mejora -- la matriz de confusion muestra en que pares de clases se equivoca mas el modelo (por ejemplo, vidrio y plastico transparente tienden a confundirse por su parecido visual).
- Existe un trade-off de negocio entre precision y velocidad: SVM con la fusion completa es el mas preciso pero tambien el mas lento de entrenar (cerca de 80 segundos en nuestras pruebas); Random Forest con HOG (F1=0.443, menos de 1 segundo) es una alternativa mas liviana si el sistema necesitara reentrenarse con frecuencia o correr en hardware limitado.
- Para el producto final, **EcoVision** recomienda partir de Fusion + SVM como motor de clasificacion, con Random Forest + HOG como alternativa rapida para escenarios con menos capacidad de computo.
